<a href="https://colab.research.google.com/github/cozen03/2026-Programming/blob/main/%ED%99%8D%ED%98%B8%EC%A4%80_%ED%9A%8C%EC%9D%98%EB%A1%9D%EB%A7%8C%EB%93%A4%EA%B8%B0_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pyannote.audio
%pip install numpy==1.26

In [ ]:
from google.colab import userdata
from huggingface_hub import login
HUGGING_FACE_TOKEN = userdata.get('HUGGING_FACE_TOKEN')
login(HUGGING_FACE_TOKEN)

In [ ]:
from google.colab import userdata
HUGGING_FACE_TOKEN = userdata.get('HUGGING_FACE_TOKEN')
# instantiate the pipeline
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  token=HUGGING_FACE_TOKEN,
)

In [ ]:
import torch

# cuda가 사용 가능한 경우 cuda를 사용하도록 설정
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))
    print('cuda is available')
else:
    print('cuda is not available')

In [ ]:
!ffmpeg -i "싼기타_비싼기타.mp3" -ar 16000 -ac 1 -c:a pcm_s16le input.wav -y

In [ ]:
# 수정된 임포트 경로
from pyannote.audio.pipelines.utils.hook import ProgressHook
import torch

# GPU 가속 설정
pipeline.to(torch.device("cuda"))

# 진행 상황을 확인하며 실행
with ProgressHook() as hook:
    diarization = pipeline("input.wav", hook=hook)

In [ ]:
from pyannote.audio.pipelines.utils.hook import ProgressHook
pipeline.to(torch.device("cuda"))
# run the pipeline on an audio file
# diarization = pipeline("audio.wav")
with ProgressHook() as hook:
    diarization = pipeline("input.wav", hook=hook)

# 1. RTTM 파일 저장
output_filename = "싼기타_비싼기타.rttm"

with open(output_filename, "w", encoding='utf-8') as rttm:
    # 핵심 수정: diarization 자체가 아니라 그 안의 'speaker_diarization'을 사용합니다.
    actual_annotation = diarization.speaker_diarization

    for turn, _, speaker in actual_annotation.itertracks(yield_label=True):
        # RTTM 형식에 맞춰 기록
        start = turn.start
        duration = turn.duration
        line = f"SPEAKER input.wav 1 {start:.3f} {duration:.3f} <NA> <NA> {speaker} <NA> <NA>\n"
        rttm.write(line)

print(f"✅ 화자 분리 결과가 성공적으로 저장되었습니다: {output_filename}")


In [ ]:
# RTTM을 CSV로 변환
import pandas as pd
rttm_path = "./싼기타_비싼기타.rttm"

df_rttm = pd.read_csv(
    rttm_path,      # rttm 파일 경로
    sep=' ',        # 구분자는 띄어쓰기
    header=None,    # 헤더는 없음
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4']
)

display(df_rttm)

In [ ]:
# start + duration을 end로 변환
df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

display(df_rttm)

In [ ]:
df_rttm["number"] = None	# number 열 만들고 None으로 초기화
df_rttm.at[0, "number"] = 0

display(df_rttm)

In [ ]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

display(df_rttm.head(10))

In [ ]:
df_rttm_grouped = df_rttm.groupby("number").agg(
    start=pd.NamedAgg(column='start', aggfunc='min'),
    end=pd.NamedAgg(column='end', aggfunc='max'),
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

In [ ]:
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

In [ ]:
df_rttm_grouped.to_csv(
    "싼기타_비싼기타_rttm.csv",
    sep=',',
    index=False
)

In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline

from google.colab import userdata
HUGGING_FACE_TOKEN = userdata.get('HUGGING_FACE_TOKEN')

def whisper_stt(
    audio_file_path: str,
    output_file_path: str = "./output.csv"
):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_id = "openai/whisper-large-v3-turbo"

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        return_timestamps=True,  # 청크별로 타임스탬프를 반환
        chunk_length_s=10,  # 입력 오디오를 10초씩 나누기
        stride_length_s=2,  # 2초씩 겹치도록 청크 나누기
    )

    result = pipe(audio_file_path)
    df = whisper_to_dataframe(result, output_file_path)

    return result, df


def whisper_to_dataframe(result, output_file_path):
    start_end_text = []

    for chunk in result["chunks"]:
        start = chunk["timestamp"][0]
        end = chunk["timestamp"][1]
        text = chunk["text"].strip()
        start_end_text.append([start, end, text])
        df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
        df.to_csv(output_file_path, index=False, sep="|")

    return df


def speaker_diarization(
        audio_file_path: str,
        output_rttm_file_path: str,
        output_csv_file_path: str
    ):
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        token=HUGGING_FACE_TOKEN
    )

    # cuda가 사용 가능한 경우 cuda를 사용하도록 설정
    if torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))
        print('cuda is available')
    else:
        print('cuda is not available')
    diarization_pipeline = pipeline(audio_file_path)

    res = pipeline(audio_file_path)

    # 2. 결과 객체 내부에 있는 실제 Annotation 데이터를 추출합니다.
    actual_annotation = res.speaker_diarization

    # 3. 추출된 Annotation 객체를 사용하여 RTTM 파일을 저장합니다.
    with open(output_rttm_file_path, "w", encoding='utf-8') as rttm:
        actual_annotation.write_rttm(rttm)

    # pandas dataframe으로 변환
    df_rttm = pd.read_csv(
        output_rttm_file_path,      # rttm 파일 경로
        sep=' ',        # 구분자는 띄어쓰기
        header=None,    # 헤더는 없음
        names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4']
    )

    df_rttm["end"] = df_rttm["start"] + df_rttm["duration"]

    # speaker_id를 기반으로 화자별로 구간을 나누기
    df_rttm["number"] = None
    df_rttm.at[0, "number"] = 0

    for i in range(1, len(df_rttm)):
        if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
            df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
        else:
            df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

    df_rttm_grouped = df_rttm.groupby("number").agg(
        start=pd.NamedAgg(column='start', aggfunc='min'),
        end=pd.NamedAgg(column='end', aggfunc='max'),
        speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
    )

    df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]

    df_rttm_grouped.to_csv(
        output_csv_file_path,
        index=False,    # 인덱스는 저장하지 않음
        encoding='utf-8'
    )
    return df_rttm_grouped


def stt_to_rttm(
        audio_file_path: str,
        stt_output_file_path: str,
        rttm_file_path: str,
        rttm_csv_file_path: str,
        final_output_csv_file_path: str
    ):

    result, df_stt = whisper_stt(
        audio_file_path,
        stt_output_file_path
    ) # ①

    df_rttm = speaker_diarization(
        audio_file_path,
        rttm_file_path,
        rttm_csv_file_path
    ) # ①

    df_rttm["text"] = "" #  ②

    for i_stt, row_stt in df_stt.iterrows(): #  ②
        overlap_dict = {}
        for i_rttm, row_rttm in df_rttm.iterrows(): # ③
            overlap = max(0, min(row_stt["end"], row_rttm["end"]) - max(row_stt["start"], row_rttm["start"]))
            overlap_dict[i_rttm] = overlap

        max_overlap = max(overlap_dict.values())
        max_overlap_idx = max(overlap_dict, key=overlap_dict.get)

        if max_overlap > 0: # ③
            df_rttm.at[max_overlap_idx, "text"] += row_stt["text"] + "\n"

    df_rttm.to_csv(
        final_output_csv_file_path,
        index=False,    # 인덱스는 저장하지 않음
        sep='|',
        encoding='utf-8'
    )  # ④
    return df_rttm


if __name__ == "__main__":
    audio_file_path = "input.wav"       # 원본 오디오 파일
    stt_output_file_path = "싼기타_비싼기타.csv"	# STT 결과 파일
    rttm_file_path = "싼기타_비싼기타.rttm"		# 화자 분리 원본 파일
    rttm_csv_file_path = "싼기타_비싼기타_rttm.csv"	# 화자 분리 CSV 파일
    final_csv_file_path = "싼기타_비싼기타_final.csv" # 최종 결과 파일

    # result, df = whisper_stt(
    #     audio_file_path,
    #     stt_output_file_path
    # )
    # print(df)

    # df_rttm = speaker_diarization(
    #     audio_file_path,
    #     rttm_file_path,
    #     rttm_csv_file_path
    # )

    # print(df_rttm)

    df_rttm = stt_to_rttm(
        audio_file_path,
        stt_output_file_path,
        rttm_file_path,
        rttm_csv_file_path,
        final_csv_file_path
    )

    print(df_rttm)



In [ ]:
import pandas as pd

meeting_note_csv_path = "싼기타_비싼기타_final.csv"
df_rttm = pd.read_csv(meeting_note_csv_path, sep='|')

display(df_rttm)

In [ ]:
# 이름 넣기
name_dict = {
    "SPEAKER_00": "AI",
    "SPEAKER_01": "이성용",
}

df_rttm["name"] = df_rttm["speaker_id"].apply(lambda x: name_dict[x])

display(df_rttm)

In [ ]:
meeting_note_txt = df_rttm[['start', 'end', 'name', 'text']].to_json(orient='records', force_ascii=False)
print(meeting_note_txt)

In [ ]:
system_prompt = f'''
너는 회의 내용을 요약하는 봇이다. 아래 회의록을 읽고, 주요 내용을 요약하라.
결과는 마크다운 형식으로 작성한다.

아래 형식에 맞추어 작성하라.

# 회의 제목

## 주요 내용

## 참석자별 입장

## 결정 사항

=============== 이하 회의록 ===============
'''

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata
key = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=key)

response = client.models.generate_content(
    model = "models/gemini-2.5-flash",
    contents=meeting_note_txt,
    config=types.GenerateContentConfig(
            temperature=0.9,
            system_instruction=system_prompt
        )
)
summary = response.text
summary = summary.strip() # 좌우 공백 제거
with open('guitar_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)
print(response.text)

In [ ]:
df_meeting_note = df_rttm[['start', 'end', 'name', 'text']].copy()
df_meeting_note.dropna(inplace=True)

meeting_note_dict = df_meeting_note.to_dict(orient='records')  # 딕셔너리 형태로 변환
meeting_note_dict